In [ ]:
import numpy as np

from scipy.fft import fft, ifft, dct, idct, dst, idst

from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, QFTGate, UnitaryGate, HGate, SwapGate
from qiskit.quantum_info import Statevector

In [2]:
WINDOW_SIZES = [256] #2^8 must n^2: 64, 128, 256, 512

def validate_window(x):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("Transform input must be 1-D")

    n = len(x)
    if n == 0 or (n & (n-1)) != 0:
        raise ValueError(f"Window length must be a power of two, got {n}")

    return x 

## Classical FFT (Fast Fourier Transforms)

In [3]:
def classical_fft(x):
    x = validate_window(x)
    return fft(x, norm="ortho")

def classical_ifft(coeffs):
    coeffs = validate_window(coeffs)
    return ifft(coeffs, norm="ortho")

## Classical DCT-II (Discrete Cosine Transforms II)

In [4]:
def classical_dct(x):
    x = validate_window(x)
    return dct(x, type=2, norm="ortho")

def classical_idct(coeffs):
    coeffs = validate_window(coeffs)
    return idct(coeffs, type=2, norm="ortho")

## Classical DST-II (Discrete Sine Transforms II)

In [5]:
def classical_dst(x):
    x = validate_window(x)
    return dst(x, type=2, norm="ortho")


def classical_idst(coeffs):
    coeffs = validate_window(coeffs)
    return idst(coeffs, type=2, norm="ortho")

## Classical DWT-Haar (Discrete Wavelet Transforms Haar)

In [6]:
def classical_haar(x):
    x = validate_window(x).astype(np.float64)
    approx = x.copy()
    details = []
    while len(approx) > 1:
        even = approx[0::2]
        odd = approx[1::2]
        next_approx = (even + odd) / np.sqrt(2.0)
        detail = (even - odd) / np.sqrt(2.0)
        details.append(detail)
        approx = next_approx

    return np.concatenate([approx, *details[::-1]])

def classical_ihaar(coeffs):
    coeffs = validate_window(coeffs).astype(np.float64)
    approx = coeffs[:1]
    offset = 1
    while offset < len(coeffs):
        n = len(approx)
        detail = coeffs[offset:offset+n]
        reconstructed = np.empty(2 * n, dtype=np.float64)
        reconstructed[0::2] = (approx + detail) / np.sqrt(2.0)
        reconstructed[1::2] = (approx - detail) / np.sqrt(2.0)
        approx = reconstructed
        offset += n

    return approx

1.3322676295501878e-15


## Quantum encodings

In [7]:
def prepare_amplitudes(x):
    x = validate_window(x)
    norm = np.linalg.norm(x)
    if norm <= 1e-15:
        return None, 0.0

    amplitudes = (x / norm).astype(np.complex128)
    num_qubits = int(np.log2(len(x)))
    qc = QuantumCircuit(num_qubits)
    qc.append(StatePreparation(amplitudes), range(num_qubits),)
    return qc, norm

## QFT

In [8]:
def quantum_fft(x):
    qc, input_norm = prepare_amplitudes(x)

    if qc is None:
        return np.zeros_like(x, dtype=np.complex128), None

    num_qubits = qc.num_qubits
    qc.append(QFTGate(num_qubits).inverse(), range(num_qubits))
    state = Statevector.from_instruction(qc).data
    coeffs = (np.asarray(state) * input_norm)
    return coeffs, qc

def quantum_ifft(coeffs):
    qc, coeff_norm = prepare_amplitudes(coeffs)

    if qc is None:
        return np.zeros_like(coeffs, dtype=np.complex128), None

    num_qubits = qc.num_qubits
    qc.append(QFTGate(num_qubits), range(num_qubits))
    state = Statevector.from_instruction(qc).data
    reconstructed = (np.asarray(state) * coeff_norm)
    return reconstructed, qc

2.1935456002220668e-13


## DCT Matrixes

In [9]:
def dct_matrix(n):
    eye = np.eye(n)
    return dct(eye, type=2, norm="ortho", axis=0)

1.2212453270876722e-15


## Quantum Matrixes Transform

In [10]:
def quantum_unitary_transform(x, matrix, label):
    qc, input_norm = prepare_amplitudes(x)

    if qc is None:
        return np.zeros_like(x, dtype=np.complex128), None

    gate = UnitaryGate(matrix, label=label,)
    qc.append(gate, range(qc.num_qubits),)
    state = Statevector.from_instruction(qc).data
    coeffs = (np.asarray(state) * input_norm)
    return coeffs, qc

def quantum_unitary_inverse(coeffs, matrix, label):
    return quantum_unitary_transform(coeffs, matrix.conj().T, label)

## QDCT-II (Quantum Discete Cosine Transform II)

In [11]:
def quantum_dct(x):
    x = validate_window(x)
    U = dct_matrix(len(x))
    return quantum_unitary_transform(x, U, "QDCT-II")

def quantum_idct(coeffs):
    coeffs = validate_window(coeffs)
    U = dct_matrix(len(coeffs))
    return quantum_unitary_inverse(coeffs, U, "IQDCT")

2.6711965972481266e-13


## QDST-II (Quantum Discrete Sine Transform II)

In [12]:
def dst_matrix(n):
    eye = np.eye(n)
    return dst(eye, type=2, norm="ortho", axis=0)

def quantum_dst(x):
    x = validate_window(x)
    U = dst_matrix(len(x))
    return quantum_unitary_transform(x, U, "QDST-II")


def quantum_idst(coeffs):
    coeffs = validate_window(coeffs)
    U = dst_matrix(len(coeffs))
    return quantum_unitary_inverse(coeffs, U, "IQDST")

2.582378755278114e-13


## DWT Haar Matrixes

In [13]:
def haar_matrix(n):
    if n == 0 or (n & (n - 1)) != 0:
        raise ValueError("Haar size must be power of two.")

    U = np.empty((n, n), dtype=np.float64)
    for j in range(n):
        basis = np.zeros(n, dtype=np.float64)
        basis[j] = 1.0
        U[:, j] = classical_haar(basis)

    return U

3.3306690738754696e-16


## QDWT (Quantum Discrete Wavelet Transform)

In [14]:
def quantum_haar(x):
    x = validate_window(x)
    U = haar_matrix(len(x))
    return quantum_unitary_transform(x, U, "QDWT-Haar")

def quantum_ihaar(coeffs):
    coeffs = validate_window(coeffs)
    U = haar_matrix(len(coeffs))
    return quantum_unitary_inverse(coeffs, U, "IQDWT-Haar")

3.1136204725612515e-13


## Main transforms process

In [15]:
CLASSICAL_TRANSFORMS = {
    "fft": (
        classical_fft,
        classical_ifft,
    ),

    "dct": (
        classical_dct,
        classical_idct,
    ),

    "dst": (
        classical_dst,
        classical_idst,
    ),

    "dwt_haar": (
        classical_haar,
        classical_ihaar,
    ),
}

QUANTUM_TRANSFORMS = {
    "qft": (
        quantum_fft,
        quantum_ifft,
    ),

    "qdct": (
        quantum_dct,
        quantum_idct,
    ),

    "qdst": (
        quantum_dst,
        quantum_idst,
    ),

    "qdwt_haar": (
        quantum_haar,
        quantum_ihaar,
    ),
}

## Sanity checks

In [16]:
rng = np.random.default_rng(42)
x = rng.normal(size=256).astype(np.float64)

In [17]:
pairs = [
    ("FFT/QFT",     classical_fft,   quantum_fft),
    ("DCT/QDCT",    classical_dct,   quantum_dct),
    ("DST/QDST",    classical_dst,   quantum_dst),
    ("Haar/QDWT",   classical_haar,  quantum_haar),

    ("IFFT/IQFT",   classical_ifft,  quantum_ifft),
    ("IDCT/IQDCT",  classical_idct,  quantum_idct),
    ("IDST/IQDST",  classical_idst,  quantum_idst),
    ("IHaar/IQDWT", classical_ihaar, quantum_ihaar),
]

for name, classical_fn, quantum_fn in pairs:
    classical_result = classical_fn(x)
    quantum_result, _ = quantum_fn(x)
    error = np.max(np.abs(classical_result - quantum_result))
    print(f"{name:12s} | max error = {error:.6e}")

FFT/QFT      | max error = 2.193546e-13
DCT/QDCT     | max error = 2.697825e-13
DST/QDST     | max error = 2.617141e-13
Haar/QDWT    | max error = 3.534611e-13
IFFT/IQFT    | max error = 2.404985e-13
IDCT/IQDCT   | max error = 2.848990e-13
IDST/IQDST   | max error = 2.238220e-13
IHaar/IQDWT  | max error = 2.972736e-13
